# Item-Item Collaborative Filtering

## Learning Objectives

1. **Explain** why item-item CF is preferred at scale over user-user CF
2. **Implement** item similarity using cosine and adjusted cosine
3. **Predict** ratings using item-based neighborhood
4. **Compare** item-item CF accuracy and computational trade-offs

## Item-Item vs User-User

**Key insight:** items tend to have more stable similarity patterns than users.

- Users' preferences drift; item genre doesn't change
- Number of items $n$ grows slower than users $m$
- Items accumulate more ratings over time → better similarity estimates

**Algorithm:**
For target user $u$ and unrated item $i$:
1. Find items $j$ that user $u$ has rated
2. Compute $\text{sim}(i, j)$ for each such $j$
3. Weighted prediction:

$$\hat{r}_{u,i} = \frac{\sum_{j \in S(i,u)} \text{sim}(i,j) \cdot r_{u,j}}{\sum_{j \in S(i,u)} |\text{sim}(i,j)|}$$

In [ ]:
import math

def adjusted_cosine(item_i, item_j, ratings):
    """Adjusted cosine: subtract user mean before computing cosine."""
    # Find users who rated both items
    users_i = {u for u, r in ratings.items() if item_i in r}
    users_j = {u for u, r in ratings.items() if item_j in r}
    common_users = users_i & users_j
    if len(common_users) < 2: return 0.0

    user_means = {u: sum(ratings[u].values())/len(ratings[u]) for u in common_users}
    ri = [ratings[u][item_i] - user_means[u] for u in common_users]
    rj = [ratings[u][item_j] - user_means[u] for u in common_users]
    num = sum(a*b for a,b in zip(ri,rj))
    den = math.sqrt(sum(a**2 for a in ri)) * math.sqrt(sum(b**2 for b in rj))
    return num/den if den > 0 else 0.0

ratings = {
    'Alice': {'M1':5,'M2':3,'M3':4,'M4':4},
    'Bob':   {'M1':3,'M2':1,'M3':2,'M4':3,'M5':3},
    'Carol': {'M1':4,'M2':3,'M3':4,'M5':5},
    'Dave':  {'M2':1,'M3':2,'M4':3,'M5':4},
    'Eve':   {'M1':1,'M3':5,'M4':2},
}
all_items = sorted(set(i for r in ratings.values() for i in r))

# Compute item-item similarity matrix
print("Adjusted Cosine Item Similarity Matrix:")
print(f"{'':>4}", end='')
for item in all_items: print(f"{item:>8}", end='')
print()
for item_i in all_items:
    print(f"{item_i:>4}", end='')
    for item_j in all_items:
        if item_i == item_j:
            print(f"{'1.000':>8}", end='')
        else:
            s = adjusted_cosine(item_i, item_j, ratings)
            print(f"{s:>8.3f}", end='')
    print()

In [ ]:
def predict_item_based(target_user, target_item, ratings, k=3):
    """Item-item CF prediction for target_user on target_item."""
    # Items rated by target user (excluding target item)
    rated_by_user = {item: r for item, r in ratings[target_user].items()
                     if item != target_item}
    # Compute similarity between target_item and each rated item
    sims = [(rated_item, adjusted_cosine(target_item, rated_item, ratings))
            for rated_item in rated_by_user]
    # Top-k similar items with positive similarity
    top_k = sorted([(j, s) for j,s in sims if s > 0], key=lambda x:-x[1])[:k]
    if not top_k:
        return sum(ratings[target_user].values())/len(ratings[target_user])
    num = sum(s * ratings[target_user][j] for j,s in top_k)
    den = sum(abs(s) for _,s in top_k)
    return num/den if den > 0 else 0

# Predict for Alice all unrated items
unrated = [i for i in all_items if i not in ratings['Alice']]
print("Item-item CF predictions for Alice:")
for item in unrated:
    pred = predict_item_based('Alice', item, ratings)
    print(f"  {item}: {pred:.3f}")

## Scalability Comparison

| Method | Similarity computation | Space | Online prediction |
|--------|----------------------|-------|------------------|
| User-user CF | $O(m^2 n)$ | $O(m^2)$ | $O(m \cdot n_u)$ |
| Item-item CF | $O(n^2 m)$ | $O(n^2)$ | $O(n_u \cdot k)$ |

For Netflix: $m=500K$ users, $n=17K$ movies.
- User-user: $500K^2 \times 17K = $ impractical
- Item-item: $17K^2 \times 500K = $ can precompute once

**Item-item CF advantages:**
- Similarity matrix fits in memory ($n^2 \ll m^2$ usually)
- Very fast online prediction (lookup precomputed similarities)
- More stable (items don't change; similarities recomputed weekly)

**Item-item CF disadvantages:**
- Doesn't improve as more users arrive (unlike user-user)
- Still struggles with very sparse utility matrices